AI Acknowledgement is provided in a separate document, including prompts and responses used to build this notebook.

In [30]:
import statsmodels.api as sm
import numpy as np
import pandas as pd

In [31]:
# Basic preparation of data as per assignment specifications.
OUTCOME_VAR = 'mdvis'
df = sm.datasets.randhie.load_pandas().data

y = np.array(df[OUTCOME_VAR])
X = df.drop(OUTCOME_VAR, axis=1)
X = sm.add_constant(X, prepend=True)

X

,const,lncoins,idp,lpi,fmde,physlm,disea,hlthg,hlthf,hlthp
0,1.0,4.615120,1,6.907755,0.000000,0.000000,13.73189,1,0,0
1,1.0,4.615120,1,6.907755,0.000000,0.000000,13.73189,1,0,0
2,1.0,4.615120,1,6.907755,0.000000,0.000000,13.73189,1,0,0
3,1.0,4.615120,1,6.907755,0.000000,0.000000,13.73189,1,0,0
4,1.0,4.615120,1,6.907755,0.000000,0.000000,13.73189,1,0,0
...,...,...,...,...,...,...,...,...,...,...
20185,1.0,0.000000,0,5.377498,0.000000,0.144292,10.57626,0,0,0
20186,1.0,0.000000,0,5.377498,0.000000,0.144292,10.57626,0,0,0
20187,1.0,3.258096,0,6.874819,8.006368,0.144292,10.57626,0,0,0
20188,1.0,3.258096,0,5.156178,6.542472,0.144292,10.57626,0,0,0


# Question 1

## a)


Starting from the Poisson probability mass function: $$𝑝(𝑦 ∣ 𝜇) = 𝑒^{−𝜇}𝜇^𝑦/𝑦!$$ derive
the negative log-likelihood $𝐽(𝜷)$ for the model above, dropping any term that does not
depend on 𝜷. Report $𝐽(𝜷_0)$.

The formula for negative log-likelihood for the Poisson regression can be found in the corresponding derivation PDF.

In [32]:
# Function for the negative log-likelihood with the probability density 
# associated with the Poisson regression provided
def j(beta, X, y):
    rolling_sum = 0

    for i in range(len(X)):
        x = X.iloc[i].values
        y_i = y[i]
        rolling_sum += np.exp(x @ beta) - y_i * (x @ beta)
    return rolling_sum

beta_0 = np.array([1, 0, 0, 0, 0, 0, 0.02, 0, 0, 0]) # "evaluate at the fixed parameter vector 𝜷0 with intercept 1, coefficient 0.02 on disea, and zeros elsewhere."

j = j(beta_0, X, y)
print(f"J(β₀): {j:.4f}")

J(β₀): -3952.0259


## b)

Obtain the gradient $∇𝐽(𝜷)$. Report two entries of $∇𝐽(𝜷_0)$: the entry for the
intercept, and the entry for hlthf.

In [33]:
# Function for the gradient of negative log-likelihood with the probability density 
# associated with the Poisson regression provided
def grad_j(beta, X, y):
    rolling_sum = np.zeros(len(beta))

    for i in range(len(X)):
        x = X.iloc[i].values
        y_i = y[i]
        rolling_sum += (np.exp(x @ beta) - y_i) * x
    return rolling_sum

gradient_j = pd.DataFrame(grad_j(beta_0, X, y), index=X.columns, columns=["Gradient"])
print(f"∇J(β₀) for intercept: \n{gradient_j.loc["const"]}\n")
print(f"∇J(β₀) for hlthf: \n{gradient_j.loc["hlthf"]}\n")

∇J(β₀) for intercept: 
Gradient    11634.636524
Name: const, dtype: float64

∇J(β₀) for hlthf: 
Gradient    14.154186
Name: hlthf, dtype: float64



## c)

Obtain the Hessian $𝒥(𝜷)$, and write it in the matrix form $𝑿^{T}𝑾𝑿$. Report the
leading diagonal entry of $𝒥(𝜷_0)$, the intercept–intercept entry.

In [34]:
# Derived from the gradient formula above
def H(beta, X, y):
    return X.T @ np.diag(np.exp(X @ beta)) @ X

hessian = pd.DataFrame(H(beta_0, X, y), index=X.columns, columns=X.columns)
print(f"𝒥(𝜷) for intercept-intercept: {hessian.loc["const", "const"]}")

𝒥(𝜷) for intercept-intercept: 69386.63652385017


# Question 2

Implement Newton’s method to minimise $𝐽$. Start from the fit with no covariates: set the
intercept to $log(\bar{𝑦})$ and every other coefficient to zero. Stop when the largest absolute change in
any coefficient falls below $10^{−10}$, allowing at least 25 iterations. Report the estimated ${𝛽}_{hlthf}$.
Newton’s method converges quadratically near the optimum, and not necessarily anywhere else,
so a starting value that already reproduces the mean of the data is worth the one line it costs.

In [35]:
# Main source is from Tutorial 4's notebook, though prompts and manual tweaks were also used.
def train_newton(x0, X, y, grad, hess, tol=1e-6, max_iter=100):

    x = x0  # Starting value
    history = [x]
    iterations = 0 
    
    for i in range(max_iter):
        iterations += 1
        x0 = x
        
        # Compute the update step by solving a linear system
        H = hess(x0, X=X, y=y)
        g = grad(x0, X=X, y=y)
        
        try:
            x = x0 - np.linalg.solve(H, g)
        except np.linalg.LinAlgError:
            # If Hessian is singular, add a small regularization
            H = H + np.eye(H.shape[0]) * 1e-8
            x = x0 - np.linalg.solve(H, g)
        
        # Store current parameters
        history.append(x)
        
        # Check convergence
        if np.max(np.abs(x-x0)) < tol:
            break
    
    print(f'Newton\'s method converged in {iterations} iterations.\n')
    return x, history 

# "Newton’s method converges quadratically near the optimum, and not necessarily anywhere else, 
# so a starting value that already reproduces the mean of the data is worth the one line it costs."
x0 = np.array([np.log(y.mean()), 0, 0, 0, 0, 0, 0, 0, 0, 0]) 

beta_hat, history = train_newton(x0, X, y, grad_j, H, tol=1e-10, max_iter=25)
beta_hat = pd.DataFrame(beta_hat, index=X.columns, columns=["Estimate"])
print(f"Estimated 𝜷 for hlthf: \n{beta_hat.loc["hlthf"]}")

Newton's method converged in 7 iterations.

Estimated 𝜷 for hlthf: 
Estimate    0.054056
Name: hlthf, dtype: float64


# Question 3

## a)

Using the estimated covariance matrix implied by your own Hessian, compute
the model-based standard errors and report the one for hlthf.

In [36]:
# Main source is from Tutorial 4's notebook, though prompts and manual tweaks were also used.
def standard_errors(beta, X, y):
    beta = np.array(beta).flatten()

    info = H(beta, X, y)
    cov = pd.DataFrame(np.linalg.inv(info), index=X.columns, columns=X.columns)
    se = pd.DataFrame(np.sqrt(np.diag(cov)), index=X.columns, columns=['SE'])
    return cov, se

cov, se = standard_errors(beta_hat, X, y)
print(f"Estimated standard error for hlthf: \n{se.loc["hlthf"]}")

Estimated standard error for hlthf: 
SE    0.01531
Name: hlthf, dtype: float64


## b)

Implement the sandwich estimator of the covariance matrix of the estimated $𝜷$, and report
the resulting robust standard error for hlthf.

In [37]:
# Main source is from Tutorial 4's notebook, though prompts and manual tweaks were also used.
def robust_standard_errors(beta, X, y):
    beta = np.array(beta).flatten()
    estimated_hessian = H(beta, X=X, y=y)
    scores = []

    for i in range(len(X)):
        x = X.iloc[i].values
        y_i = y[i]

        score_i = (np.exp(x @ beta) - y_i) * x
        scores.append(score_i)

    scores = np.array(scores)

    estimated_G = scores.T @ scores

    robust_cov = pd.DataFrame(np.linalg.inv(estimated_hessian) @ estimated_G @ np.linalg.inv(estimated_hessian), index=X.columns, columns=X.columns)
    robust_se = pd.DataFrame(np.sqrt(np.diag(robust_cov)), index=X.columns, columns=['Robust SE'])
    return robust_cov, robust_se

robust_cov, robust_se = robust_standard_errors(beta_hat, X, y)
print(f"Estimated sandwich standard error for hlthf: \n{robust_se.loc["hlthf"]}")

Estimated sandwich standard error for hlthf: 
Robust SE    0.042478
Name: hlthf, dtype: float64


## c)

Compute the ratio of the robust standard error to the model-based standard
error for all ten coefficients. Report the smallest and the largest.

In [38]:
# Main source is from Tutorial 4's notebook, though prompts and manual tweaks were also used.
def compute_se_ratio(model_se, robust_se):
    se_ratio = model_se['SE'] / robust_se['Robust SE']
    se_ratio_df = pd.DataFrame(se_ratio, index=model_se.index, columns=['SE Ratio'])
    return se_ratio_df

se_ratio_df = compute_se_ratio(se, robust_se)
print("Ratio of model-based SE to robust SE:")
for index, row in se_ratio_df.iterrows():
    print(f"  {index}: {row['SE Ratio']}")

print(f"Minimum ratio: predictor {se_ratio_df['SE Ratio'].idxmin()} with {se_ratio_df['SE Ratio'].min()}")
print(f"Maximum ratio: predictor {se_ratio_df['SE Ratio'].idxmax()} with {se_ratio_df['SE Ratio'].max()}")

Ratio of model-based SE to robust SE:
  const: 0.39094954502308676
  lncoins: 0.4002761332982152
  idp: 0.3956452964398333
  lpi: 0.39687139408387584
  fmde: 0.389849011326529
  physlm: 0.37007441081995085
  disea: 0.3581394155756076
  hlthg: 0.4125276973189606
  hlthf: 0.36041596559464767
  hlthp: 0.34125314744275936
Minimum ratio: predictor hlthp with 0.34125314744275936
Maximum ratio: predictor hlthg with 0.4125276973189606


## d)

Report the Wald 𝑧-statistic for hlthf computed two ways: with the model-based
standard error, and with the robust one.

In [39]:
model_wald_stats = (np.array(beta_hat).flatten() / se["SE"]) ** 2
robust_wald_stats = (np.array(beta_hat).flatten() / robust_se["Robust SE"]) ** 2

print(f"Wald z-stat for hlthf using model-based SE: \n{model_wald_stats.loc["hlthf"]}\n")
print(f"Wald z-stat for hlthf using sandwich-based SE: \n{robust_wald_stats.loc["hlthf"]}\n")

Wald z-stat for hlthf using model-based SE: 
12.466658653957953

Wald z-stat for hlthf using sandwich-based SE: 
1.6194148234039734



# Question 4

For $𝑏 = 1, … , 𝐵$ with 𝐵 = 1000: draw 𝑛 row indices with replacement; refit the Poisson regression
on the resampled rows $(𝒙𝑖, 𝑦𝑖)$; store the estimated $𝜷^{∗(𝑏)}$. Report the bootstrap standard error for hlthf.
You may refit with statsmodels rather than your own Newton routine. Instantiate a fresh
generator with rng = np.random.default_rng(3830), and report the standard error as the
sample standard deviation of the 𝐵 replicate coefficients, computed with ddof=1.

In [40]:
def bootstrap_train(X, y, max_iter=1000, random_state=3830):
    rng = np.random.default_rng(3830)

    bootstrap_beta_hats = []
    for iteration in range(max_iter):
        # Resample the data with replacement
        bootstrap_indices = rng.choice(len(X), size=len(X), replace=True)
        X_bootstrap = X.iloc[bootstrap_indices]
        y_bootstrap = y[bootstrap_indices]

        # Fit the model on the bootstrap sample
        beta_hat_bootstrap = sm.GLM(y_bootstrap, X_bootstrap, family=sm.families.Poisson()).fit().params
        bootstrap_beta_hats.append(beta_hat_bootstrap)

    bootstrap_beta_hats = np.array(bootstrap_beta_hats)
    bootstrap_se = pd.DataFrame(np.std(bootstrap_beta_hats, axis=0, ddof=1), index=X.columns, columns=['Bootstrap SE'])

    return bootstrap_beta_hats, bootstrap_se

bootstrap_beta_hats, bootstrap_se = bootstrap_train(X, y, max_iter=1000, random_state=3830)
bootstrap_se

,Bootstrap SE
const,0.028065
lncoins,0.007262
idp,0.025838
lpi,0.004587
fmde,0.004176
physlm,0.033584
disea,0.001556
hlthg,0.023126
hlthf,0.040977
hlthp,0.075907


# Question 5

No computation is explicitly required here, so this is mainly just for verification and analysis.

In [41]:
def compare_se_ratios(*std_errors):
    ratios = []
    unions = []
    index = []
    for outer_type in std_errors:

        for inner_type in std_errors:
            set_to_add = set([f"{outer_type.columns[-1]}", f"{inner_type.columns[-1]}"])
            if not set_to_add in unions:
                ratios.append(np.array(outer_type).flatten()/np.array(inner_type).flatten())

                index.append(f"{outer_type.columns[-1]}-{inner_type.columns[-1]}")
                unions.append(set_to_add)

    return pd.DataFrame(ratios, index=index, columns=std_errors[0].index).T

ratios = compare_se_ratios(se, robust_se, bootstrap_se)
ratios

,SE-SE,SE-Robust SE,SE-Bootstrap SE,Robust SE-Robust SE,Robust SE-Bootstrap SE,Bootstrap SE-Bootstrap SE
const,1.0,0.390950,0.397742,1.0,1.017373,1.0
lncoins,1.0,0.400276,0.397157,1.0,0.992209,1.0
idp,1.0,0.395645,0.410913,1.0,1.038590,1.0
lpi,1.0,0.396871,0.398563,1.0,1.004262,1.0
fmde,1.0,0.389849,0.386237,1.0,0.990736,1.0
physlm,1.0,0.370074,0.364429,1.0,0.984744,1.0
disea,1.0,0.358139,0.362878,1.0,1.013232,1.0
hlthg,1.0,0.412528,0.400013,1.0,0.969663,1.0
hlthf,1.0,0.360416,0.373617,1.0,1.036628,1.0
hlthp,1.0,0.341253,0.346205,1.0,1.014510,1.0


In [42]:
phi_estimates = (1/(ratios["SE-Robust SE"])) ** 2

print(f"Estimated 𝜙: {phi_estimates.mean()}")

Estimated 𝜙: 6.936069504809535
